# QRT Challenge — Submission RF V1

- **Train** : `features_V1.csv` (X_train_sample + y_train_sample + features engineerées)
- **Test** : `Data/X_test.csv` (feature engineering appliqué ici)
- **Modèle** : Random Forest avec les meilleurs hyperparamètres de `benchmark_models.ipynb`
- **Sortie** : `submission_rf_V1.csv`

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

SEED = 42
print('Libraries loaded.')

Libraries loaded.


## 1. Best Params (depuis benchmark_models)

⚠️ **Copie ici les `best_params` du notebook `benchmark_models.ipynb`** si les valeurs sont différentes.

In [3]:
import json
with open('best_params_rf.json', 'r') as f:
    best_params = json.load(f)
print('Best params importés:', best_params)

Best params importés: {'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 15}


## 2. Feature Engineering (fonction commune train/test)

In [4]:
def feature_engineering(df):
    """
    Applique exactement le même feature engineering que features_V1.
    Fonctionne sur train ET test (sans dépendre de la colonne target).
    """
    data = df.copy()
    
    ret_cols = [f'RET_{i}' for i in range(1, 21)]
    
    # Interaction
    data['RET_1_TURNOVER'] = data['RET_1'] * data['MEDIAN_DAILY_TURNOVER']
    
    # Rolling statistics
    data['RET_MEAN_5'] = data[[f'RET_{i}' for i in range(1, 6)]].mean(axis=1)
    data['RET_MEAN_20'] = data[ret_cols].mean(axis=1)
    data['RET_STD_5'] = data[[f'RET_{i}' for i in range(1, 6)]].std(axis=1)
    data['RET_STD_20'] = data[ret_cols].std(axis=1)
    
    # Cumulative return
    data['RET_CUM_5'] = data[[f'RET_{i}' for i in range(1, 6)]].sum(axis=1)
    data['RET_CUM_20'] = data[ret_cols].sum(axis=1)
    
    # Momentum indicators
    data['RET_POSITIVE_COUNT_5'] = (data[[f'RET_{i}' for i in range(1, 6)]] > 0).sum(axis=1)
    data['RET_POSITIVE_COUNT_20'] = (data[ret_cols] > 0).sum(axis=1)
    
    # Recent vs old
    data['RET_RECENT_VS_OLD'] = data['RET_MEAN_5'] - data[[f'RET_{i}' for i in range(16, 21)]].mean(axis=1)
    
    return data

print('Feature engineering function defined.')

Feature engineering function defined.


## 3. Normalisation par groupe

In [5]:
def normalize_by_group(X_df, group_col='GROUP'):
    """Standardize features within each group."""
    X_norm = X_df.copy()
    numeric_cols = [c for c in X_norm.columns if c != group_col]
    
    scalers = {}  # on garde les scalers pour le test
    for grp in X_norm[group_col].unique():
        mask = X_norm[group_col] == grp
        scaler = StandardScaler()
        X_norm.loc[mask, numeric_cols] = scaler.fit_transform(X_norm.loc[mask, numeric_cols])
        scalers[grp] = scaler
    
    return X_norm, scalers

def normalize_test_by_group(X_df, scalers, group_col='GROUP'):
    """Normalise le test avec les scalers du train."""
    X_norm = X_df.copy()
    numeric_cols = [c for c in X_norm.columns if c != group_col]
    
    for grp in X_norm[group_col].unique():
        mask = X_norm[group_col] == grp
        if grp in scalers:
            X_norm.loc[mask, numeric_cols] = scalers[grp].transform(X_norm.loc[mask, numeric_cols])
        else:
            # Groupe inconnu -> fit_transform
            scaler = StandardScaler()
            X_norm.loc[mask, numeric_cols] = scaler.fit_transform(X_norm.loc[mask, numeric_cols])
    
    return X_norm

print('Normalization functions defined.')

Normalization functions defined.


## 4. Préparer le Train

In [6]:
# Charger features_V1 (train)
df_train = pd.read_csv('features_V1.csv')
print(f'Train brut : {df_train.shape}')

# Target
y_train = (df_train['target'] > 0).astype(int)

# Feature columns
exclude_cols = ['ROW_ID', 'TS', 'ALLOCATION', 'target']
feature_cols = [c for c in df_train.columns if c not in exclude_cols]

X_train = df_train[feature_cols].fillna(0)

# Normalisation par groupe (on garde les scalers)
X_train_norm, train_scalers = normalize_by_group(X_train)

print(f'X_train : {X_train_norm.shape}')
print(f'Features : {feature_cols}')

Train brut : (50000, 56)
X_train : (50000, 52)
Features : ['RET_20', 'RET_19', 'RET_18', 'RET_17', 'RET_16', 'RET_15', 'RET_14', 'RET_13', 'RET_12', 'RET_11', 'RET_10', 'RET_9', 'RET_8', 'RET_7', 'RET_6', 'RET_5', 'RET_4', 'RET_3', 'RET_2', 'RET_1', 'SIGNED_VOLUME_20', 'SIGNED_VOLUME_19', 'SIGNED_VOLUME_18', 'SIGNED_VOLUME_17', 'SIGNED_VOLUME_16', 'SIGNED_VOLUME_15', 'SIGNED_VOLUME_14', 'SIGNED_VOLUME_13', 'SIGNED_VOLUME_12', 'SIGNED_VOLUME_11', 'SIGNED_VOLUME_10', 'SIGNED_VOLUME_9', 'SIGNED_VOLUME_8', 'SIGNED_VOLUME_7', 'SIGNED_VOLUME_6', 'SIGNED_VOLUME_5', 'SIGNED_VOLUME_4', 'SIGNED_VOLUME_3', 'SIGNED_VOLUME_2', 'SIGNED_VOLUME_1', 'MEDIAN_DAILY_TURNOVER', 'GROUP', 'RET_1_TURNOVER', 'RET_MEAN_5', 'RET_MEAN_20', 'RET_STD_5', 'RET_STD_20', 'RET_CUM_5', 'RET_CUM_20', 'RET_POSITIVE_COUNT_5', 'RET_POSITIVE_COUNT_20', 'RET_RECENT_VS_OLD']


## 5. Préparer le Test (même feature engineering)

In [7]:
# Charger X_test brut
X_test_raw = pd.read_csv('Data/X_test.csv')
print(f'Test brut : {X_test_raw.shape}')

# Appliquer le même feature engineering
X_test_fe = feature_engineering(X_test_raw)
print(f'Test après FE : {X_test_fe.shape}')

# Garder les ROW_ID pour la submission
test_row_ids = X_test_fe['ROW_ID']

# Sélectionner les mêmes features que le train
X_test = X_test_fe[feature_cols].fillna(0)

# Normaliser avec les scalers du train
X_test_norm = normalize_test_by_group(X_test, train_scalers)

print(f'X_test normalisé : {X_test_norm.shape}')

Test brut : (31870, 45)
Test après FE : (31870, 55)
X_test normalisé : (31870, 52)


## 6. Entraîner le Random Forest & Prédire

In [8]:
# Entraîner sur tout le train
print('Training Random Forest...')
rf = RandomForestClassifier(**best_params, random_state=SEED, n_jobs=-1)
rf.fit(X_train_norm, y_train)

# Vérification sur le train
train_acc = rf.score(X_train_norm, y_train)
print(f'Train accuracy : {train_acc:.4f}')

# Prédire sur le test
y_pred = rf.predict(X_test_norm)
print(f'\nPrédictions test :')
print(f'  Total : {len(y_pred)}')
print(f'  Up (1) : {(y_pred == 1).sum()} ({(y_pred == 1).mean()*100:.1f}%)')
print(f'  Down (0) : {(y_pred == 0).sum()} ({(y_pred == 0).mean()*100:.1f}%)')

Training Random Forest...
Train accuracy : 0.9346

Prédictions test :
  Total : 31870
  Up (1) : 18457 (57.9%)
  Down (0) : 13413 (42.1%)


## 7. Créer le fichier de submission

In [9]:
submission = pd.DataFrame({
    'ROW_ID': test_row_ids,
    'prediction': y_pred
})

submission.to_csv('submission_rf_V1.csv', index=False)

print(f'Submission sauvegardée : submission_rf_V1.csv')
print(f'Shape : {submission.shape}')
print(f'\nAperçu :')
print(submission.head(10))

# Vérification format
sample = pd.read_csv('sample_submission.csv')
assert list(submission.columns) == list(sample.columns), 'Colonnes incorrectes !'
assert len(submission) == len(sample), f'Taille incorrecte : {len(submission)} vs {len(sample)}'
print(f'\n✅ Format vérifié (même colonnes et taille que sample_submission.csv)')

Submission sauvegardée : submission_rf_V1.csv
Shape : (31870, 2)

Aperçu :
   ROW_ID  prediction
0  527073           1
1  527074           1
2  527075           1
3  527076           1
4  527077           1
5  527078           1
6  527079           1
7  527080           0
8  527081           1
9  527082           0

✅ Format vérifié (même colonnes et taille que sample_submission.csv)
